In [1]:
import pandas as pd
import numpy as np
import re

# 1. Load dataset
df = pd.read_csv("https://raw.githubusercontent.com/swapnilsaurav/Dataset/refs/heads/master/data_quality_training.csv")

print("Original shape:", df.shape)

# Keep original copy
dirty_df = df.copy()

# 2. Remove duplicate records - Uniqueness
df = df.drop_duplicates()

# 3. Standardize text columns - Consistency
city_mapping = {
    "Hyd": "Hyderabad", "HYD": "Hyderabad",
    "BLR": "Bengaluru", "Bangalore": "Bengaluru", "bengaluru": "Bengaluru",
    "MUM": "Mumbai", "Bombay": "Mumbai", "mumbai": "Mumbai",
    "DEL": "Delhi", "New Delhi": "Delhi", "delhi": "Delhi",
    "CHE": "Chennai", "Madras": "Chennai", "chennai": "Chennai"
}

payment_mapping = {
    "paid": "Paid", "PAID": "Paid", "P": "Paid", "payment_done": "Paid",
    "Pending": "Pending",
    "Failed": "Failed",
    "Refunded": "Refunded"
}

product_mapping = {
    "MOB": "Mobile",
    "LPT": "Laptop",
    "HD": "Headphones",
    "MSE": "Mouse",
    "KEY": "Keyboard",
    "MON": "Monitor",
    "PRN": "Printer",
    "SWT": "Smartwatch",
    "TBL": "Tablet"
}

df["City"] = df["City"].replace(city_mapping)
df["Payment_Status"] = df["Payment_Status"].replace(payment_mapping)
df["Product_Category"] = df["Product_Category"].replace(product_mapping)

# 4. Fix accuracy issues

# Unrealistic age
df.loc[(df["Age"] < 0) | (df["Age"] > 100), "Age"] = np.nan

# Very high or negative order amount
df.loc[(df["Order_Amount"] < 0) | (df["Order_Amount"] > 200000), "Order_Amount"] = np.nan

# City-State validation
city_state_mapping = {
    "Hyderabad": "Telangana",
    "Mumbai": "Maharashtra",
    "Pune": "Maharashtra",
    "Bengaluru": "Karnataka",
    "Chennai": "Tamil Nadu",
    "Delhi": "Delhi",
    "Kolkata": "West Bengal",
    "Ahmedabad": "Gujarat",
    "Jaipur": "Rajasthan",
    "Lucknow": "Uttar Pradesh"
}

df["Expected_State"] = df["City"].map(city_state_mapping)

df.loc[
    df["Expected_State"].notna(),
    "State"
] = df["Expected_State"]

df.drop(columns=["Expected_State"], inplace=True)

# 5. Fix completeness issues

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Order_Amount"] = df["Order_Amount"].fillna(df["Order_Amount"].median())
df["City"] = df["City"].fillna("Unknown")
df["Customer_Name"] = df["Customer_Name"].fillna("Unknown Customer")
df["Email"] = df["Email"].fillna("missing_email@example.com")

# 6. Date parsing - Interpretability and Timeliness

df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce", dayfirst=False)
df["Delivery_Date"] = pd.to_datetime(df["Delivery_Date"], errors="coerce", dayfirst=False)
df["Last_Updated"] = pd.to_datetime(df["Last_Updated"], errors="coerce", dayfirst=False)

# Fill missing order date with median valid order date
median_order_date = df["Order_Date"].median()
df["Order_Date"] = df["Order_Date"].fillna(median_order_date)

# Future order dates
today = pd.Timestamp.today().normalize()
df.loc[df["Order_Date"] > today, "Order_Date"] = today

# Delivery date before order date
df.loc[df["Delivery_Date"] < df["Order_Date"], "Delivery_Date"] = df["Order_Date"] + pd.Timedelta(days=3)

# Missing delivery date
df["Delivery_Date"] = df["Delivery_Date"].fillna(df["Order_Date"] + pd.Timedelta(days=3))

# Last updated before order date
df.loc[df["Last_Updated"] < df["Order_Date"], "Last_Updated"] = df["Order_Date"]

# Missing last updated
df["Last_Updated"] = df["Last_Updated"].fillna(df["Delivery_Date"])

# 7. Email validation - Interpretability

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

def fix_email(email, customer_id):
    email = str(email)
    if re.match(email_pattern, email):
        return email
    return f"{str(customer_id).lower()}@example.com"

df["Email"] = df.apply(
    lambda row: fix_email(row["Email"], row["Customer_ID"]),
    axis=1
)

# 8. Phone validation - Believability

def fix_phone(phone):
    phone = str(phone)
    phone = re.sub(r"\D", "", phone)
    
    if len(phone) == 10 and phone[0] in "6789":
        return phone
    else:
        return "9999999999"

df["Phone"] = df["Phone"].apply(fix_phone)

# 9. Final formatting

df["Order_Date"] = df["Order_Date"].dt.strftime("%Y-%m-%d")
df["Delivery_Date"] = df["Delivery_Date"].dt.strftime("%Y-%m-%d")
df["Last_Updated"] = df["Last_Updated"].dt.strftime("%Y-%m-%d %H:%M:%S")

# 10. Remove helper issue columns if required

# Optional: keep these columns for teaching
# df = df.drop(columns=["Data_Quality_Issue", "Issue_Description"])

# 11. Save cleaned dataset

df.to_csv("cleaned_data_quality_training.csv", index=False)

print("Cleaned shape:", df.shape)
print("Cleaned file saved as: cleaned_data_quality_training.csv")

# 12. Data quality checks after cleaning

print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nDuplicate rows after cleaning:")
print(df.duplicated().sum())

print("\nAge range:")
print(df["Age"].min(), "to", df["Age"].max())

print("\nOrder amount range:")
print(df["Order_Amount"].min(), "to", df["Order_Amount"].max())

print("\nUnique cities:")
print(df["City"].unique())

print("\nUnique payment statuses:")
print(df["Payment_Status"].unique())

print("\nUnique product categories:")
print(df["Product_Category"].unique())

Original shape: (1000, 18)
Cleaned shape: (979, 18)
Cleaned file saved as: cleaned_data_quality_training.csv

Missing values after cleaning:
Order_ID              0
Customer_ID           0
Customer_Name         0
Age                   0
Gender                0
Email                 0
Phone                 0
City                  0
State                 0
Product_Category      0
Order_Date            0
Delivery_Date         0
Order_Amount          0
Payment_Status        0
Source_System         0
Last_Updated          0
Data_Quality_Issue    0
Issue_Description     0
dtype: int64

Duplicate rows after cleaning:
0

Age range:
0.0 to 65.0

Order amount range:
1.0 to 59999.0

Unique cities:
['Delhi' 'Chennai' 'Mumbai' 'Bengaluru' 'Pune' 'Kolkata' 'Lucknow'
 'Ahmedabad' 'Unknown' 'Hyderabad' 'Jaipur']

Unique payment statuses:
['Paid' 'Refunded' 'Pending' 'Failed']

Unique product categories:
['Tablet' 'Monitor' 'Router' 'Keyboard' 'Mouse' 'Mobile' 'Smartwatch'
 'Headphones' 'Printer' 'Lapt